In [ ]:
import torch
import torch.nn.functional as F
from torchvision import datasets
import numpy as np
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchvision.ops import sigmoid_focal_loss
import transformers
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, project_root)
import importlib
import models.clip_model as clip
importlib.reload(clip)
import utilities.util_metrics as metrics
importlib.reload(metrics)
import utilities.util_dataloader as dataloader
importlib.reload(dataloader)


In [ ]:

#Build
directory = r'C:\Users\frank\OneDrive\Documents\vsCodeProjects\cytology\data'
device = "cuda" if torch.cuda.is_available() else "cpu"

temp_dataset = datasets.ImageFolder(root=directory)
num_classes = len(temp_dataset.classes)

model, preprocess = clip.build_clip_with_head(
    num_classes=num_classes,
    device=device
)

In [ ]:
#data
batch_size = 16
validation_split = 0.4
seed = 123

full_dataset = datasets.ImageFolder(
    root=directory,
    transform=preprocess
)

# Split
val_len = int(len(full_dataset) * validation_split)
train_len = len(full_dataset) - val_len

torch.manual_seed(seed)
train_dataset, val_dataset = random_split(full_dataset, [train_len, val_len])

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [ ]:
#Compile
model.train()
# optimizer

optimizer = torch.optim.Adam([
    {"params": model.model.parameters(), "lr": 3e-5},
    {"params": model.classifier.parameters(), "lr": 3e-4}
])

weights = torch.tensor([1.0, 1.5, 1.0]).to(device)


scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=30
)

In [ ]:
#training loop

num_epochs = 30

for epoch in range(num_epochs):
    
    #training
    model.train()
    train_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        ce_loss = F.cross_entropy(outputs, labels, reduction='none')
        probs = torch.softmax(outputs, dim=1)
        pt = probs.gather(1, labels.unsqueeze(1)).squeeze()

        loss = ((1 - pt) ** 1.5) * ce_loss
        loss = loss.mean()

        train_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    scheduler.step()

    # validation
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    thresholds = torch.tensor([0.5, 0.55, 0.5]).to(device)

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            
            loss = F.cross_entropy(outputs, labels)
            val_loss += loss.item()

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs / thresholds, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    # averages
    train_loss /= len(train_loader)
    val_loss /= len(val_loader)
    val_acc = correct / total if total > 0 else 0

    print(f"Epoch {epoch+1}: "
          f"Train Loss={train_loss:.4f}, "
          f"Val Loss={val_loss:.4f}, "
          f"Val Acc={val_acc:.4f}")

In [ ]:
#grid search
all_logits = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = model(images)

        all_logits.append(outputs.cpu())
        all_labels.append(labels)

logits = torch.cat(all_logits)
labels = torch.cat(all_labels)

probs = torch.softmax(logits, dim=1)

best_f1 = 0
best_t = 0

for t in np.linspace(0.40, 0.70, 50):
    thresholds = torch.tensor([0.5, t, 0.5])

    preds = torch.argmax(probs / thresholds, dim=1)

    f1 = metrics.f1_score(labels, preds, average='weighted')

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best LSIL threshold:", best_t)
print("Best F1:", best_f1)

In [ ]:
model.eval()

y_pred = []
y_true = []
y_pred_probs = []

thresholds = torch.tensor([0.5, best_t, 0.5]).to(device)

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)

        preds = torch.argmax(probs / thresholds, dim=1) 

        # store results
        y_pred.extend(preds.cpu().numpy())
        y_true.extend(labels.numpy())
        y_pred_probs.extend(probs.cpu().numpy())

# Convert to numpy arrays
y_pred = np.array(y_pred)
y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)

print("Shapes:")
print("y_true:", y_true.shape)
print("y_pred:", y_pred.shape)
print("y_pred_probs:", y_pred_probs.shape)
#results 
import json


acc = metrics.compute_accuracy(y_true, y_pred)
f1 = metrics.compute_f1(y_true, y_pred)
auc_score = metrics.compute_auc(y_true, y_pred_probs, num_classes=3)

print("Accuracy:", acc)
print("F1 Score:", f1)
print("AUC:", auc_score)

class_names = ["NILM", "LSIL", "HSIL"]

metrics.plot_confusion_matrix(y_true, y_pred, class_names)
metrics.plot_roc_multiclass(y_true, y_pred_probs, 3, class_names)

results = {
    "model_name": "clip", 
    "accuracy": acc,
    "f1_score": f1,
    "auc": auc_score
}

print(json.dumps(results))

In [ ]:
#results 
acc = metrics.compute_accuracy(y_true, y_pred)
f1 = metrics.compute_f1(y_true, y_pred)
auc_score = metrics.compute_auc(y_true, y_pred_probs, num_classes=3)

print("Accuracy:", acc)
print("F1 Score:", f1)
print("AUC:", auc_score)

class_names = ["NILM", "LSIL", "HSIL"]

metrics.plot_confusion_matrix(y_true, y_pred, class_names)
metrics.plot_roc_multiclass(y_true, y_pred_probs, 3, class_names)